In [2]:
# from comet_ml import Experiment
# from comet_ml.integration.pytorch import log_model

import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn, optim
import torch.nn.functional as F
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import pandas as pd
import pickle as pkl

import matplotlib.pyplot as plt
import numpy as np
import io, os
from tqdm import tqdm

from PIL import Image
from torchvision import models 
from torchvision.models import resnet18

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import KBinsDiscretizer
import statsmodels.api as sm

import lightning.pytorch as pl
import torch
import torch.nn as nn
import torchmetrics
import torchvision
from torchvision import models

from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay, multilabel_confusion_matrix

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

## Data Module

In [4]:

class ImageSequenceDataModule(pl.LightningDataModule):
    """
        Pytorch Lightning DataModule for Image+Sequence dataset. This will download the dataset, prepare data loaders and apply
        data augmentation.
    """
    def __init__(self, curve_dict_path, target_df_path, batch_size=32, shuffle=True, num_workers =16, igi_call=False):
        super().__init__()
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.num_workers = num_workers
        self.igi_call = igi_call

        print("WE ARE USING THE IMAGE SEQUENCE DATASET")

        with open(curve_dict_path, 'rb') as file:
            self.curve_dict = pkl.load(file)
        self.target_df = pd.read_csv(target_df_path)

        self.target_df['igi_fp'] = (self.target_df['Igi_call_quant'] > self.target_df['groundtruth_target']).astype(int)
        self.target_df['igi_fn'] = (self.target_df['Igi_call_quant'] < self.target_df['groundtruth_target']).astype(int)

        self.target_df_train = self.target_df[self.target_df['split']=='train']
        self.curve_dict_train = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df_train['curve_idx'].values}
        
        self.target_df_val = self.target_df[self.target_df['split']=='val']
        self.curve_dict_val = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df_val['curve_idx'].values}

        self.target_df_test = self.target_df[self.target_df['split']=='test']
        self.curve_dict_test = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df_test['curve_idx'].values}

        mean_list = []
        std_list = []

        for key, curve in tqdm(self.curve_dict_train.items()):
            mean_curve = np.array(curve).mean().item()
            std_curve = np.array(curve).std().item()

            mean_list.append(mean_curve)
            std_list.append(std_curve)

        self.norm_mean = np.array(mean_list).mean().item()
        self.norm_std = np.array(std_list).mean().item()

    def prepare_data(self):
        return

    def setup(self, stage=None):
        self.train = ImageSequenceDataset(self.curve_dict_train, self.target_df_train, igi_call=self.igi_call, mean=self.norm_mean, std=self.norm_std)
        self.val = ImageSequenceDataset(self.curve_dict_val, self.target_df_val, igi_call=self.igi_call, mean=self.norm_mean, std=self.norm_std)
        self.test = ImageSequenceDataset(self.curve_dict_test, self.target_df_test, igi_call=self.igi_call, mean=self.norm_mean, std=self.norm_std)

    def train_dataloader(self):
        return DataLoader(self.train, batch_size=self.batch_size, shuffle=True, num_workers = self.num_workers)

    def val_dataloader(self):
        return DataLoader(self.val, batch_size=self.batch_size, shuffle=False, num_workers = self.num_workers)
    
    def test_dataloader(self):
        return DataLoader(self.test, batch_size=self.batch_size, shuffle=False, num_workers = self.num_workers)

class ImageSequenceDataset(Dataset):
    def __init__(self, curve_dict, target_df, img_directory = 'data/curve_imgs/', sequence_len=40, igi_call=True,
                 mean=0, std=1):
        self.curve_dict = curve_dict
        self.target_df = target_df

        #one-hot encode gene indicator
        self.one_hot = pd.get_dummies(self.target_df['target'], prefix='target')
        self.target_df = pd.concat([self.target_df, self.one_hot], axis=1)

        self.img_directory = img_directory
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.sequence_len = sequence_len
        self.igi_call = igi_call

        self.mean = mean
        self.std = std

        # Image transformations: Resize and Normalize

        self.img_transforms = transforms.Compose([
            transforms.Lambda(lambda image: image.convert('RGB')),
            transforms.Resize((224, 224)),  # Resizing to a consistent size
            transforms.ToTensor(),  # Convert PIL image to tensor
            transforms.Normalize((0.5,), (0.5,))  # Normalizing to [0,1]
            ])

        # self.img_transforms = transforms.Compose([
        #     transforms.Lambda(lambda image: image.convert('RGB')),
        #     transforms.Resize((128, 128)),  # Resizing to a consistent size
        #     transforms.ToTensor(),  # Convert PIL image to tensor
        #     transforms.Normalize((0.5,), (0.5,))  # Normalizing to [0,1]
        #     ])
   
    def __len__(self):
        return len(self.curve_dict.keys())
    
    def __getitem__(self, idx):
        curve_idx = list(self.curve_dict.keys())[idx]

        # Image processing
        curve_img_path = os.path.join(self.img_directory, f'curve_{curve_idx}.png')
        curve_img = Image.open(curve_img_path)
        curve_img = self.img_transforms(curve_img)

        #sequence processing
        sequence = self.curve_dict[curve_idx][:self.sequence_len]
        #TODO fix normalization to normalizing by mean and std of sequences in train set
        sequence = torch.tensor(sequence, dtype=torch.float32)
        sequence_normalized = (sequence - torch.tensor(self.mean, dtype=torch.float32)) / torch.tensor(self.std, dtype=torch.float32)

        row = self.target_df.loc[self.target_df['curve_idx'] == curve_idx]
        target = torch.tensor(row['groundtruth_target'].values[0], dtype=torch.float)

        if self.igi_call:
            igi_fp = torch.tensor(row['igi_fp'].values[0], dtype=torch.float)
            igi_fn = torch.tensor(row['igi_fn'].values[0], dtype=torch.float)
            target = torch.stack([target, igi_fp, igi_fn], dim=0)

        return (curve_img, sequence_normalized.unsqueeze(1)), target
    
class ImageDataModule(pl.LightningDataModule):
    """
        Pytorch Lightning DataModule for Image+Sequence dataset. This will download the dataset, prepare data loaders and apply
        data augmentation.
    """
    def __init__(self, curve_dict_path, target_df_path, img_directory, batch_size=32, shuffle=True, num_workers=4, igi_call=False):
        super().__init__()
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.num_workers = num_workers
        self.igi_call = igi_call
        self.img_directory = img_directory
        
        print("WE ARE USING THE IMAGE DATASET")

        with open(curve_dict_path, 'rb') as file:
            self.curve_dict = pkl.load(file)
        self.target_df = pd.read_csv(target_df_path)

        self.target_df['igi_fp'] = (self.target_df['Igi_call_quant'] > self.target_df['groundtruth_target']).astype(int)
        self.target_df['igi_fn'] = (self.target_df['Igi_call_quant'] < self.target_df['groundtruth_target']).astype(int)

        self.target_df_train = self.target_df[self.target_df['split']=='train']
        self.curve_dict_train = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df_train['curve_idx'].values}
        
        self.target_df_val = self.target_df[self.target_df['split']=='val']
        self.curve_dict_val = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df_val['curve_idx'].values}

        self.target_df_test = self.target_df[self.target_df['split']=='test']
        self.curve_dict_test = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df_test['curve_idx'].values}

        mean_list = []
        std_list = []

        for key, curve in tqdm(self.curve_dict_train.items()):
            mean_curve = np.array(curve).mean().item()
            std_curve = np.array(curve).std().item()

            mean_list.append(mean_curve)
            std_list.append(std_curve)

        self.norm_mean = np.array(mean_list).mean().item()
        self.norm_std = np.array(std_list).mean().item()

        self.setup()

    def prepare_data(self):
        return

    def setup(self, stage=None):
        self.train = ImageDataset(self.curve_dict_train, self.target_df_train, self.img_directory, igi_call=self.igi_call, mean=self.norm_mean, std=self.norm_std)
        self.val = ImageDataset(self.curve_dict_val, self.target_df_val, self.img_directory, igi_call=self.igi_call, mean=self.norm_mean, std=self.norm_std)
        self.test = ImageDataset(self.curve_dict_test, self.target_df_test, self.img_directory, igi_call=self.igi_call, mean=self.norm_mean, std=self.norm_std)

    def train_dataloader(self):
        return DataLoader(self.train, batch_size=self.batch_size, shuffle=True, num_workers = self.num_workers)

    def val_dataloader(self):
        return DataLoader(self.val, batch_size=self.batch_size, shuffle=False, num_workers = self.num_workers)
    
    def test_dataloader(self):
        return DataLoader(self.test, batch_size=self.batch_size, shuffle=False, num_workers = self.num_workers)

class ImageDataset(Dataset):
    def __init__(self, curve_dict, target_df, img_directory = '../data/curve_imgs/', sequence_len=40, igi_call=True,
                 mean=0, std=1):
        
        self.curve_dict = curve_dict
        self.target_df = target_df

        #one-hot encode gene indicator
        self.one_hot = pd.get_dummies(self.target_df['target'], prefix='target')
        self.target_df = pd.concat([self.target_df, self.one_hot], axis=1)

        self.img_directory = img_directory
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.sequence_len = sequence_len

        self.mean = mean
        self.std = std
        self.igi_call = igi_call

        # Image transformations: Resize and Normalize
        self.img_transforms = transforms.Compose([
            transforms.Lambda(lambda image: image.convert('RGB')),
            transforms.Resize((224, 224)),  # Resizing to a consistent size
            transforms.ToTensor(),  # Convert PIL image to tensor
            transforms.Normalize((0.5,), (0.5,))  # Normalizing to [0,1]
            ])
   
    def __len__(self):
        return len(self.curve_dict.keys())
    
    def __getitem__(self, idx):
        curve_idx = list(self.curve_dict.keys())[idx]

        # Image processing
        curve_img_path = os.path.join(self.img_directory, f'curve_{curve_idx}.png')
        curve_img = Image.open(curve_img_path)
        curve_img = self.img_transforms(curve_img)

        #gene info processing
        row = self.target_df.loc[self.target_df['curve_idx'] == curve_idx]

        target = torch.tensor(row['groundtruth_target'].values[0], dtype=torch.float)

        if self.igi_call:
            igi_fp = torch.tensor(row['igi_fp'].values[0], dtype=torch.float)
            igi_fn = torch.tensor(row['igi_fn'].values[0], dtype=torch.float)
            target = torch.stack([target, igi_fp, igi_fn], dim=0)

        return curve_img, target, curve_idx

class ImageSequenceGeneDataModule(pl.LightningDataModule):
    """
        Pytorch Lightning DataModule for Image+Sequence dataset. This will download the dataset, prepare data loaders and apply
        data augmentation.
    """
    def __init__(self, curve_dict_path, target_df_path, img_directory, batch_size=32, shuffle=True, num_workers=4, igi_call=False):
        super().__init__()
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.num_workers = num_workers
        self.igi_call = igi_call
        self.img_directory = img_directory

        print("WE ARE USING THE IMAGE SEQUENCE GENE DATASET")

        with open(curve_dict_path, 'rb') as file:
            self.curve_dict = pkl.load(file)
        
        self.target_df = pd.read_csv(target_df_path)

        self.target_df['igi_fp'] = (self.target_df['Igi_call_quant'] > self.target_df['groundtruth_target']).astype(int)
        self.target_df['igi_fn'] = (self.target_df['Igi_call_quant'] < self.target_df['groundtruth_target']).astype(int)

        self.target_df_train = self.target_df[self.target_df['split']=='train']
        self.curve_dict_train = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df_train['curve_idx'].values}
        
        self.target_df_val = self.target_df[self.target_df['split']=='val']
        self.curve_dict_val = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df_val['curve_idx'].values}

        self.target_df_test = self.target_df[self.target_df['split']=='test']
        self.curve_dict_test = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df_test['curve_idx'].values}

        mean_list = []
        std_list = []

        for key, curve in tqdm(self.curve_dict_train.items()):
            mean_curve = np.array(curve).mean().item()
            std_curve = np.array(curve).std().item()

            mean_list.append(mean_curve)
            std_list.append(std_curve)

        self.norm_mean = np.array(mean_list).mean().item()
        if np.isnan(self.norm_mean):
            self.norm_mean = 155626.8370536778
        self.norm_std = np.array(std_list).mean().item()
        if np.isnan(self.norm_std):
            self.norm_std = 94477.0057018847

        self.setup()

    def prepare_data(self):
        return

    def setup(self, stage=None):
        self.train = ImageSequenceGeneDataset(self.curve_dict_train, self.target_df_train, self.img_directory, igi_call=self.igi_call, mean=self.norm_mean, std=self.norm_std)
        self.val = ImageSequenceGeneDataset(self.curve_dict_val, self.target_df_val, self.img_directory, igi_call=self.igi_call, mean=self.norm_mean, std=self.norm_std)
        self.test = ImageSequenceGeneDataset(self.curve_dict_test, self.target_df_test, self.img_directory, igi_call=self.igi_call, mean=self.norm_mean, std=self.norm_std)

    def train_dataloader(self):
        return DataLoader(self.train, batch_size=self.batch_size, shuffle=True, num_workers = self.num_workers)

    def val_dataloader(self):
        return DataLoader(self.val, batch_size=self.batch_size, shuffle=False, num_workers = self.num_workers)
    
    def test_dataloader(self):
        return DataLoader(self.test, batch_size=self.batch_size, shuffle=False, num_workers = self.num_workers)


class ImageSequenceGeneDataset(Dataset):
    def __init__(self, curve_dict, target_df, img_directory = 'data/curve_imgs/', sequence_len=40, igi_call=False,
                 mean=0, std=1):
        self.curve_dict = curve_dict
        self.target_df = target_df

        #one-hot encode gene indicator
        
        target_ls = ['target_' + t for t in ['S gene','N gene','E gene','RnaseP','MS2','ORF1ab']]

        self.one_hot = pd.get_dummies(self.target_df['target'], prefix='target')
        
        if self.one_hot.shape[1] != 6:
            for target in target_ls:
                if target not in self.one_hot.columns:
                    self.one_hot[target] = False
        
        self.target_df = pd.concat([self.target_df, self.one_hot], axis=1)

        self.img_directory = img_directory
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.sequence_len = sequence_len

        self.mean = mean
        self.std = std
        self.igi_call = igi_call

        # Image transformations: Resize and Normalize
        self.img_transforms = transforms.Compose([
            transforms.Lambda(lambda image: image.convert('RGB')),
            transforms.Resize((224, 224)),  # Resizing to a consistent size
            transforms.ToTensor(),  # Convert PIL image to tensor
            transforms.Normalize((0.5,), (0.5,))  # Normalizing to [0,1]
            ])
   
    def __len__(self):
        return len(self.curve_dict.keys())
    
    def __getitem__(self, idx):
        curve_idx = list(self.curve_dict.keys())[idx]

        # Image processing
        curve_img_path = os.path.join(self.img_directory, f'curve_{curve_idx}.png')
        curve_img = Image.open(curve_img_path)
        curve_img = self.img_transforms(curve_img)

        #sequence processing
        sequence = self.curve_dict[curve_idx][:self.sequence_len]
        #TODO fix normalization to normalizing by mean and std of sequences in train set
        sequence = torch.tensor(sequence, dtype=torch.float32)
        sequence_normalized = (sequence - torch.tensor(self.mean, dtype=torch.float32)) / torch.tensor(self.std, dtype=torch.float32)

        #gene info processing
        row = self.target_df.loc[self.target_df['curve_idx'] == curve_idx]
        gene_type = torch.tensor(row[self.one_hot.columns].values, dtype=torch.float32)

        target = torch.tensor(row['groundtruth_target'].values[0], dtype=torch.float)

        if self.igi_call:
            igi_fp = torch.tensor(row['igi_fp'].values[0], dtype=torch.float)
            igi_fn = torch.tensor(row['igi_fn'].values[0], dtype=torch.float)
            target = torch.stack([target, igi_fp, igi_fn], dim=0)

        return curve_img, sequence_normalized.unsqueeze(1), gene_type.squeeze(1), target, curve_idx

class ViTFusionDataset(Dataset):
    def __init__(self, curve_dict, target_df,img_directory = 'data/curve_imgs/', split='train', sequence_len=40, 
                 mean=0, std=1):
        self.curve_dict = curve_dict
        self.target_df = target_df

        #one-hot encode gene indicator
        target_ls = ['target_' + t for t in ['S gene','N gene','E gene','RnaseP','MS2','ORF1ab']]

        self.one_hot = pd.get_dummies(self.target_df['target'], prefix='target')
        
        if self.one_hot.shape[1] != 6:
            for target in target_ls:
                if target not in self.one_hot.columns:
                    self.one_hot[target] = False
        
        self.target_df = pd.concat([self.target_df, self.one_hot], axis=1)

        self.img_directory = img_directory
        self.split = split
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.sequence_len = sequence_len

        self.mean = mean
        self.std = std

        # Image transformations: Resize and Normalize
        self.img_transforms = transforms.Compose([
            transforms.Lambda(lambda image: image.convert('RGB')),
            transforms.Resize((224, 224)),  # Resizing to a consistent size
            transforms.ToTensor(),  # Convert PIL image to tensor
            transforms.Normalize((0.5,), (0.5,))  # Normalizing to [0,1]
            ])

        #Implementation of train test split
        if self.split == 'train':
            self.target_df = self.target_df[self.target_df['split']=='train']
            self.curve_dict = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df['curve_idx'].values}
        elif self.split == 'val':
            self.target_df = self.target_df[self.target_df['split']=='val']
            self.curve_dict = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df['curve_idx'].values}
        elif self.split == 'test':
            self.target_df = self.target_df[self.target_df['split']=='test']
            self.curve_dict = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df['curve_idx'].values}
        else:
            raise NotImplementedError
        
    def __len__(self):
        return len(self.curve_dict.keys())
    
    def __getitem__(self, idx):
        curve_idx = list(self.curve_dict.keys())[idx]
        # Image processing
        curve_img_path = os.path.join(self.img_directory, f'curve_{curve_idx}.png')
        curve_img = Image.open(curve_img_path)
        curve_img = self.img_transforms(curve_img)

        #sequence processing
        sequence = self.curve_dict[curve_idx][:self.sequence_len]
        #TODO fix normalization to normalizing by mean and std of sequences in train set
        sequence = torch.tensor(sequence, dtype=torch.float32)
        sequence_normalized = (sequence - torch.tensor(self.mean, dtype=torch.float32)) / torch.tensor(self.std, dtype=torch.float32)

        #gene info processing
        row = self.target_df.loc[self.target_df['curve_idx'] == curve_idx]
        gene_type = torch.tensor(row[self.one_hot.columns].values, dtype=torch.float32)

        #target data retrieval
        # Extract values from the dataframe
        target = torch.tensor(row['groundtruth_target'].values[0], dtype=torch.long)
        igi_fp = torch.tensor(row['igi_fp'].values[0], dtype=torch.long)
        igi_fn = torch.tensor(row['igi_fn'].values[0], dtype=torch.long)

        # Create a 3-dimensional vector
        vector = [target, igi_fp, igi_fn]
        #target = self.target_df.loc[self.target_df['curve_idx'] == curve_idx, 'groundtruth_target'].values[0]

        return curve_img, sequence_normalized, gene_type, vector, curve_idx


## Model Definition

In [5]:
class Classifier(pl.LightningModule):
    def __init__(self, num_classes=2, init_lr=1e-4):
        super().__init__()
        self.init_lr = init_lr
        self.num_classes = num_classes

        # Define loss fn for classifier
        self.loss = nn.BCELoss()

        self.accuracy = torchmetrics.Accuracy(task="binary" if self.num_classes == 2 else "multiclass", num_classes=self.num_classes)
        self.auc = torchmetrics.AUROC(task="binary" if self.num_classes == 2 else "multiclass", num_classes=self.num_classes)

        self.training_outputs = []
        self.validation_outputs = []

    def get_xy(self, batch):
        x, y = batch[0], batch[1]
        return x, y

    def training_step(self, batch, batch_idx):
        x, y = self.get_xy(batch)

        ## TODO: get predictions from your model and store them as y_hat
        y_hat = self.forward(*x)
        #y_hat = self.forward(x)
        loss = sum(self.loss(y_hat[:,i],y[:,i]) for i in range(3))

        self.log('train_loss', loss, prog_bar=True, sync_dist=True)

        ## Store the predictions and labels for use at the end of the epoch
        self.training_outputs.append({
            "y_hat": y_hat,
            "y": y
        })
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = self.get_xy(batch)

        y_hat = self.forward(*x)
        #y_hat = self.forward(x)
        loss = sum(self.loss(y_hat[:,i],y[:,i]) for i in range(3))

        self.log('val_loss', loss, prog_bar=True, sync_dist=True)

        self.validation_outputs.append({
            "y_hat": y_hat,
            "y": y
        })
        return loss

    def test_step(self, batch, batch_idx):
        x, y = self.get_xy(batch)

        y_hat = self.forward(*x)

        #loss = self.loss(y_hat,y)
        loss = sum(self.loss(y_hat[:,i],y[:,i]) for i in range(3))

        self.log('test_loss', loss, sync_dist=True, prog_bar=True)
        self.log('test_acc', self.accuracy(y_hat, y), sync_dist=True, prog_bar=True)

        self.test_outputs.append({
            "y_hat": y_hat,
            "y": y
        })
        return loss
    
    def on_train_epoch_end(self):
        y_hat = torch.cat([o["y_hat"] for o in self.training_outputs])
        y = torch.cat([o["y"] for o in self.training_outputs])
        
        self.log("train_auc", self.auc(y_hat, y), sync_dist=True, prog_bar=True)
        self.log("train_acc", self.accuracy(y_hat, y), sync_dist=True, prog_bar=True)
        self.training_outputs = []

    def on_validation_epoch_end(self):
        y_hat = torch.cat([o["y_hat"] for o in self.validation_outputs])
        y = torch.cat([o["y"] for o in self.validation_outputs])
        
        self.log("val_auc", self.auc(y_hat, y), sync_dist=True, prog_bar=True)
        self.log("val_acc", self.accuracy(y_hat, y), sync_dist=True, prog_bar=True)
        #self.validation_outputs = []
        
        # save to process later for evaluation
        torch.save(y_hat, 'y_hat_val_image.pt')
        torch.save(y, 'y_val_true.pt')
    
    def on_test_epoch_end(self):
        y_hat = torch.cat([o["y_hat"] for o in self.test_outputs])
        y = torch.cat([o["y"] for o in self.test_outputs])

        if self.num_classes == 2:
            probs = F.softmax(y_hat, dim=-1)[:,-1]
        else:
            probs = F.softmax(y_hat, dim=-1)

        self.log("test_auc", self.auc(probs, y.view(-1)), sync_dist=True, prog_bar=True)

        self.log("val_auc", self.auc(y_hat, y), sync_dist=True, prog_bar=True)
        self.log("val_acc", self.accuracy(y_hat, y), sync_dist=True, prog_bar=True)
        self.test_outputs = []

        # save to process later for evaluation
        torch.save(y_hat, 'y_hat_test_image.pt')
        torch.save(y, 'y_test_true.pt')

    def configure_optimizers(self):
        ## TODO: Define your optimizer and learning rate scheduler here (hint: Adam is a good default)

        optimizer = torch.optim.Adam(self.parameters(), lr=self.init_lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer)

        # return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'monitor':'val_loss'}}
        return optimizer

class FusionModel(Classifier):
    """
        Model that takes in sequence and image data and outputs single prediction head.
    """
    def __init__(self, input_size=1, hidden_size=512, latent_dim=512, sequence_length=40, num_layers=5, init_lr=1e-4, pretrained=True):
        super().__init__(num_classes=2, init_lr=init_lr)
        self.save_hyperparameters()

        self.latent_dim = latent_dim
        
        # Image processing via EfficientNet_V2_L
        # TODO change to true
        # self.effnet = models.efficientnet_v2_l(pretrained=True)
        # num_ftrs = self.effnet.classifier[1].in_features
        # self.effnet.classifier = nn.Linear(num_ftrs, self.latent_dim)  # Adjusting to output a 512-dimensional 

        if pretrained:
            self.vit = models.vit_b_32(weights='IMAGENET1K_V1')
        else:
            self.vit = models.vit_b_32(pretrained=False)
        num_ftrs = self.vit.num_classes
        self.vit_classifier = nn.Linear(num_ftrs, self.latent_dim)  # Adjusting to output a 512-dimensional 

        # Sequence processing via LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.hidden_state = (torch.zeros(num_layers, sequence_length, hidden_size), torch.zeros(num_layers, sequence_length, hidden_size))
        
        # Final fully connected layer to ensure the LSTM output has a size of 512
        self.lstm_fc = nn.Linear(hidden_size, self.latent_dim)

        # Caluclate neural_net input size after appending genes
        neural_net_input = self.latent_dim*2

        # Fusion of image and sequence representations
        self.fc = nn.Sequential(
            nn.Linear(neural_net_input, 512),  # Concatenated vectors are of size 1024 (512 from image + 512 from sequence)
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, image, sequence):
        # Image processing
        img_latent = self.vit_classifier(self.vit(image))

        # Sequence processing
        lstm_out, _ = self.lstm(sequence)
        seq_latent = self.lstm_fc(lstm_out[:, -1, :])  # Taking the last output from LSTM for the whole sequence

        # Fusion
        fusion = torch.cat((img_latent, seq_latent), dim=1)
        output = self.fc(fusion)

        return output.squeeze()


class GeneFusionModel(Classifier):
    """
        Model that takes in sequence, image, and gene data and outputs one prediction head.
    """
    def __init__(self, input_size=1, hidden_size=512, latent_dim=512, sequence_length=40, num_layers=5, genes=6, delta=64, num_heads=1, init_lr=1e-4):
        super().__init__(num_classes=2, init_lr=init_lr)
        self.save_hyperparameters()

        self.latent_dim = latent_dim
        self.delta = delta
        
        self.vit = models.vit_b_32(weights='IMAGENET1K_V1')
        num_ftrs = self.vit.num_classes
        self.vit_classifier = nn.Linear(num_ftrs, self.latent_dim)  # Adjusting to output a 512-dimensional 

        # Sequence processing via LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.hidden_state = (torch.zeros(num_layers, sequence_length, hidden_size), torch.zeros(num_layers, sequence_length, hidden_size))
        # Final fully connected layer to ensure the LSTM output has a size of 512
        self.lstm_fc = nn.Linear(hidden_size, self.latent_dim)

        # Delta Sequence processing via LSTM
        self.lstm_delta = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.hidden_state_delta = (torch.zeros(num_layers, sequence_length-1, hidden_size), torch.zeros(num_layers, sequence_length-1, hidden_size))
        # Final fully connected layer to ensure the LSTM output has a size of 512
        self.lstm_fc_delta = nn.Linear(hidden_size, self.latent_dim)

        # Caluclate neural_net input size after appending genes and delta latent
        neural_net_input = self.latent_dim*3 + genes

        # Fusion of image and sequence representations
        self.fc = nn.Sequential(
            nn.Linear(neural_net_input, 512),  # Concatenated vectors are of size 1024 (512 from image + 512 from sequence)
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
            )

    def forward(self, image, sequence, genes):
        # Image processing
        img_latent = self.vit_classifier(self.vit(image))

        # Sequence processing
        lstm_out, _ = self.lstm(sequence)
        seq_latent = self.lstm_fc(lstm_out[:, -1, :])  # Taking the last output from LSTM for the whole sequence

        # Calculating delta
        delta_seq = sequence[:, 1:] - sequence[:, :-1] #taking first difference
        lstm_out_delta, _ = self.lstm_delta(delta_seq)
        seq_latent_delta = self.lstm_fc_delta(lstm_out_delta[:, -1, :])  # Taking the last output from LSTM for the whole sequence

        # Fusion
        fusion = torch.cat((img_latent, seq_latent, genes, seq_latent_delta), dim=1)
        output = self.fc(fusion)

        return output.squeeze()

class SeqModel(Classifier):
    def __init__(self, input_size=1, hidden_size=512, latent_dim=512, sequence_length=40, num_layers=5, genes=6, delta=64, num_heads=3, init_lr=1e-4):
        super().__init__(num_classes=2, init_lr=init_lr)
        self.save_hyperparameters()

        self.latent_dim = latent_dim
        self.delta = delta

        # Sequence processing via LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.hidden_state = (torch.zeros(num_layers, sequence_length, hidden_size), torch.zeros(num_layers, sequence_length, hidden_size))
        # Final fully connected layer to ensure the LSTM output has a size of 512
        self.lstm_fc = nn.Linear(hidden_size, self.latent_dim)

        # Caluclate neural_net input size after appending genes and delta latent
        neural_net_input = self.latent_dim

        # Fusion of image and sequence representations
        self.fc = nn.Sequential(
            nn.Linear(neural_net_input, 512),  # Concatenated vectors are of size 1024 (512 from image + 512 from sequence)
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            # nn.Linear(64, 1),
            # nn.Sigmoid()
            )

        # Prediction heads
        self.heads = nn.ModuleList([nn.Linear(64, 1) for _ in range(num_heads)])

    def forward(self, image, sequence, genes):
        # Sequence processing
        lstm_out, _ = self.lstm(sequence)
        seq_latent = self.lstm_fc(lstm_out[:, -1, :])  # Taking the last output from LSTM for the whole sequence

        # Fusion
        fusion = seq_latent
        # fusion = torch.cat((seq_latent), dim=1)
        # fusion = torch.cat((img_latent, seq_latent, genes.squeeze(1), seq_latent_delta), dim=1)
        output = self.fc(fusion)

        # Get predictions for each head
        outputs = torch.stack([torch.sigmoid(head(output)) for head in self.heads], dim=-1)

        return outputs.squeeze()
    
    def on_validation_epoch_end(self):
        y_hat = torch.cat([o["y_hat"] for o in self.validation_outputs])
        y = torch.cat([o["y"] for o in self.validation_outputs])

        self.log("val_auc", self.auc(y_hat[:,0], y[:,0]), sync_dist=True, prog_bar=True)
        self.log("val_acc", self.accuracy(y_hat[:,0], y[:,0]), sync_dist=True, prog_bar=True)
        self.validation_outputs = []

class SeqDeltaModel(Classifier):
    def __init__(self, input_size=1, hidden_size=512, latent_dim=512, sequence_length=40, num_layers=5, genes=6, delta=64, num_heads=3, init_lr=1e-4):
        super().__init__(num_classes=2, init_lr=init_lr)
        self.save_hyperparameters()

        self.latent_dim = latent_dim
        self.delta = delta

        # Sequence processing via LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.hidden_state = (torch.zeros(num_layers, sequence_length, hidden_size), torch.zeros(num_layers, sequence_length, hidden_size))
        # Final fully connected layer to ensure the LSTM output has a size of 512
        self.lstm_fc = nn.Linear(hidden_size, self.latent_dim)

        # Caluclate neural_net input size after appending genes and delta latent
        neural_net_input = self.latent_dim + self.delta

        # Fusion of image and sequence representations
        self.fc = nn.Sequential(
            nn.Linear(neural_net_input, 512),  # Concatenated vectors are of size 1024 (512 from image + 512 from sequence)
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            # nn.Linear(64, 1),
            # nn.Sigmoid()
            )

        # Prediction heads
        self.heads = nn.ModuleList([nn.Linear(64, 1) for _ in range(num_heads)])

    def forward(self, image, sequence, genes):
        # Sequence processing
        lstm_out, _ = self.lstm(sequence)
        seq_latent = self.lstm_fc(lstm_out[:, -1, :])  # Taking the last output from LSTM for the whole sequence

        # Calculating delta
        delta_latent = torch.max(sequence, dim=1)[0] - torch.min(sequence, dim=1)[0]
        delta_latent = delta_latent.expand((-1, self.delta))

        # Fusion
        fusion = torch.cat((seq_latent, delta_latent), dim=1)

        output = self.fc(fusion)

        # Get predictions for each head
        outputs = torch.stack([torch.sigmoid(head(output)) for head in self.heads], dim=-1)

        return outputs.squeeze()
    
    def on_validation_epoch_end(self):
        y_hat = torch.cat([o["y_hat"] for o in self.validation_outputs])
        y = torch.cat([o["y"] for o in self.validation_outputs])

        self.log("val_auc", self.auc(y_hat[:,0], y[:,0]), sync_dist=True, prog_bar=True)
        self.log("val_acc", self.accuracy(y_hat[:,0], y[:,0]), sync_dist=True, prog_bar=True)
        self.validation_outputs = []

class SeqCurveModel(Classifier):
    def __init__(self, input_size=1, hidden_size=512, latent_dim=512, sequence_length=40, num_layers=5, genes=6, delta=64, num_heads=3, init_lr=1e-4, pretrained=True):
        super().__init__(num_classes=2, init_lr=init_lr)
        self.save_hyperparameters()

        self.latent_dim = latent_dim
        self.delta = delta
        self.pretrained = pretrained
        
        if self.pretrained:
            self.vit = models.vit_b_32(weights='IMAGENET1K_V1')
        else:
            self.vit = models.vit_b_32(pretrained=False)

        num_ftrs = self.vit.num_classes
        self.vit_classifier = nn.Linear(num_ftrs, self.latent_dim)  

        # Sequence processing via LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.hidden_state = (torch.zeros(num_layers, sequence_length, hidden_size), torch.zeros(num_layers, sequence_length, hidden_size))
        # Final fully connected layer to ensure the LSTM output has a size of 512
        self.lstm_fc = nn.Linear(hidden_size, self.latent_dim)

        # Caluclate neural_net input size after appending genes and delta latent
        neural_net_input = self.latent_dim*2

        # Fusion of image and sequence representations
        self.fc = nn.Sequential(
            nn.Linear(neural_net_input, 512),  # Concatenated vectors are of size 1024 (512 from image + 512 from sequence)
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            # nn.Linear(64, 1),
            # nn.Sigmoid()
            )

        # Prediction heads
        self.heads = nn.ModuleList([nn.Linear(64, 1) for _ in range(num_heads)])

    def forward(self, image, sequence, genes):
        # Sequence processing
        lstm_out, _ = self.lstm(sequence)
        seq_latent = self.lstm_fc(lstm_out[:, -1, :])  # Taking the last output from LSTM for the whole sequence

        # Image processing
        img_latent = self.vit_classifier(self.vit(image))

        # Fusion
        fusion = torch.cat((img_latent, seq_latent), dim=1)

        output = self.fc(fusion)

        # Get predictions for each head
        outputs = torch.stack([torch.sigmoid(head(output)) for head in self.heads], dim=-1)

        return outputs.squeeze()
    
    def on_validation_epoch_end(self):
        y_hat = torch.cat([o["y_hat"] for o in self.validation_outputs])
        y = torch.cat([o["y"] for o in self.validation_outputs])

        self.log("val_auc", self.auc(y_hat[:,0], y[:,0]), sync_dist=True, prog_bar=True)
        self.log("val_acc", self.accuracy(y_hat[:,0], y[:,0]), sync_dist=True, prog_bar=True)
        self.validation_outputs = []

class SeqDeltaGeneModel(Classifier):
    def __init__(self, input_size=1, hidden_size=512, latent_dim=512, sequence_length=40, num_layers=5, genes=6, delta=64, num_heads=3, init_lr=1e-4):
        super().__init__(num_classes=2, init_lr=init_lr)
        self.save_hyperparameters()

        self.latent_dim = latent_dim
        self.delta = delta

        # Sequence processing via LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.hidden_state = (torch.zeros(num_layers, sequence_length, hidden_size), torch.zeros(num_layers, sequence_length, hidden_size))
        # Final fully connected layer to ensure the LSTM output has a size of 512
        self.lstm_fc = nn.Linear(hidden_size, self.latent_dim)

        # Caluclate neural_net input size after appending genes and delta latent
        neural_net_input = self.latent_dim + self.delta + genes

        # Fusion of image and sequence representations
        self.fc = nn.Sequential(
            nn.Linear(neural_net_input, 512),  # Concatenated vectors are of size 1024 (512 from image + 512 from sequence)
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            # nn.Linear(64, 1),
            # nn.Sigmoid()
            )

        # Prediction heads
        self.heads = nn.ModuleList([nn.Linear(64, 1) for _ in range(num_heads)])

    def forward(self, image, sequence, genes):
        # Sequence processing
        lstm_out, _ = self.lstm(sequence)
        seq_latent = self.lstm_fc(lstm_out[:, -1, :])  # Taking the last output from LSTM for the whole sequence

        # Calculating delta
        delta_latent = torch.max(sequence, dim=1)[0] - torch.min(sequence, dim=1)[0]
        delta_latent = delta_latent.expand((-1, self.delta))

        # Fusion
        fusion = torch.cat((seq_latent, genes.squeeze(1), delta_latent), dim=1)

        output = self.fc(fusion)

        # Get predictions for each head
        outputs = torch.stack([torch.sigmoid(head(output)) for head in self.heads], dim=-1)

        return outputs.squeeze()
    
    def on_validation_epoch_end(self):
        y_hat = torch.cat([o["y_hat"] for o in self.validation_outputs])
        y = torch.cat([o["y"] for o in self.validation_outputs])

        self.log("val_auc", self.auc(y_hat[:,0], y[:,0]), sync_dist=True, prog_bar=True)
        self.log("val_acc", self.accuracy(y_hat[:,0], y[:,0]), sync_dist=True, prog_bar=True)
        self.validation_outputs = []

class SeqGeneModel(Classifier):
    def __init__(self, input_size=1, hidden_size=512, latent_dim=512, sequence_length=40, num_layers=5, genes=6, delta=64, num_heads=3, init_lr=1e-4):
        super().__init__(num_classes=2, init_lr=init_lr)
        self.save_hyperparameters()

        self.latent_dim = latent_dim
        self.delta = delta

        # Sequence processing via LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.hidden_state = (torch.zeros(num_layers, sequence_length, hidden_size), torch.zeros(num_layers, sequence_length, hidden_size))
        # Final fully connected layer to ensure the LSTM output has a size of 512
        self.lstm_fc = nn.Linear(hidden_size, self.latent_dim)

        # Caluclate neural_net input size after appending genes and delta latent
        neural_net_input = self.latent_dim + genes

        # Fusion of image and sequence representations
        self.fc = nn.Sequential(
            nn.Linear(neural_net_input, 512),  # Concatenated vectors are of size 1024 (512 from image + 512 from sequence)
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            # nn.Linear(64, 1),
            # nn.Sigmoid()
            )

        # Prediction heads
        self.heads = nn.ModuleList([nn.Linear(64, 1) for _ in range(num_heads)])

    def forward(self, image, sequence, genes):
        # Sequence processing
        lstm_out, _ = self.lstm(sequence)
        seq_latent = self.lstm_fc(lstm_out[:, -1, :])  # Taking the last output from LSTM for the whole sequence

        # Fusion
        # fusion = seq_latent
        # print('Seq size:', seq_latent.size())
        fusion = torch.cat((seq_latent, genes.squeeze(1)), dim=1)
        # fusion = torch.cat((img_latent, seq_latent, genes.squeeze(1), seq_latent_delta), dim=1)
        output = self.fc(fusion)

        # Get predictions for each head
        outputs = torch.stack([torch.sigmoid(head(output)) for head in self.heads], dim=-1)

        return outputs.squeeze()
    
    def on_validation_epoch_end(self):
        y_hat = torch.cat([o["y_hat"] for o in self.validation_outputs])
        y = torch.cat([o["y"] for o in self.validation_outputs])

        self.log("val_auc", self.auc(y_hat[:,0], y[:,0]), sync_dist=True, prog_bar=True)
        self.log("val_acc", self.accuracy(y_hat[:,0], y[:,0]), sync_dist=True, prog_bar=True)
        self.validation_outputs = []

class SeqCurveGeneModel(Classifier):
    def __init__(self, input_size=1, hidden_size=512, latent_dim=512, sequence_length=40, num_layers=5, genes=6, delta=64, num_heads=3, init_lr=1e-4, pretrained=True):
        super().__init__(num_classes=2, init_lr=init_lr)
        self.save_hyperparameters()

        self.latent_dim = latent_dim
        self.delta = delta
        self.pretrained = pretrained
        
        if self.pretrained:
            self.vit = models.vit_b_32(weights='IMAGENET1K_V1')
        else:
            self.vit = models.vit_b_32(pretrained=False)

        num_ftrs = self.vit.num_classes
        self.vit_classifier = nn.Linear(num_ftrs, self.latent_dim)  

        # Sequence processing via LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.hidden_state = (torch.zeros(num_layers, sequence_length, hidden_size), torch.zeros(num_layers, sequence_length, hidden_size))
        # Final fully connected layer to ensure the LSTM output has a size of 512
        self.lstm_fc = nn.Linear(hidden_size, self.latent_dim)

        # Caluclate neural_net input size after appending genes and delta latent
        neural_net_input = self.latent_dim*2 + genes

        # Fusion of image and sequence representations
        self.fc = nn.Sequential(
            nn.Linear(neural_net_input, 512),  # Concatenated vectors are of size 1024 (512 from image + 512 from sequence)
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            # nn.Linear(64, 1),
            # nn.Sigmoid()
            )

        # Prediction heads
        self.heads = nn.ModuleList([nn.Linear(64, 1) for _ in range(num_heads)])

    def forward(self, image, sequence, genes):
        # Sequence processing
        lstm_out, _ = self.lstm(sequence)
        seq_latent = self.lstm_fc(lstm_out[:, -1, :])  # Taking the last output from LSTM for the whole sequence

        # Image processing
        img_latent = self.vit_classifier(self.vit(image))

        # Fusion
        fusion = torch.cat((img_latent, seq_latent, genes.squeeze(1)), dim=1)

        output = self.fc(fusion)

        # Get predictions for each head
        outputs = torch.stack([torch.sigmoid(head(output)) for head in self.heads], dim=-1)

        return outputs.squeeze()
    
    def on_validation_epoch_end(self):
        y_hat = torch.cat([o["y_hat"] for o in self.validation_outputs])
        y = torch.cat([o["y"] for o in self.validation_outputs])

        self.log("val_auc", self.auc(y_hat[:,0], y[:,0]), sync_dist=True, prog_bar=True)
        self.log("val_acc", self.accuracy(y_hat[:,0], y[:,0]), sync_dist=True, prog_bar=True)
        self.validation_outputs = []

class CurveShapeModel(Classifier):
    """
        Model that solely finetunes the ViT model from the sequence.
    """
    def __init__(self, input_size=1, hidden_size=512, latent_dim=512, sequence_length=40, num_layers=5, genes=6, delta=64, num_heads=3, init_lr=1e-4, pretrained=True):
        super().__init__(num_classes=2, init_lr=init_lr)
        self.save_hyperparameters()

        self.latent_dim = latent_dim
        self.delta = delta
        self.pretrained = pretrained
        
        if self.pretrained:
            self.vit = models.vit_b_32(weights='IMAGENET1K_V1')
        else:
            self.vit = models.vit_b_32(pretrained=False)

        num_ftrs = self.vit.num_classes
        self.vit_classifier = nn.Linear(num_ftrs, self.latent_dim)  

        # Fusion of image and sequence representations
        self.fc = nn.Sequential(
            nn.Linear(self.latent_dim, 512),  # Concatenated vectors are of size 1024 (512 from image + 512 from sequence)
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            # nn.Linear(64, 1),
            # nn.Sigmoid()
            )

        # Prediction heads
        self.heads = nn.ModuleList([nn.Linear(64, 1) for _ in range(num_heads)])

    def forward(self, image):
        # Image processing
        img_latent = self.vit_classifier(self.vit(image))
        output = self.fc(img_latent)

        # Get predictions for each head
        outputs = torch.stack([torch.sigmoid(head(output)) for head in self.heads], dim=-1)

        return outputs.squeeze()

    def on_validation_epoch_end(self):
        y_hat = torch.cat([o["y_hat"] for o in self.validation_outputs])
        y = torch.cat([o["y"] for o in self.validation_outputs])

        self.log("val_auc", self.auc(y_hat[:,0], y[:,0]), sync_dist=True, prog_bar=True)
        self.log("val_acc", self.accuracy(y_hat[:,0], y[:,0]), sync_dist=True, prog_bar=True)
        self.validation_outputs = []

class CurveShapeDeltaModel(Classifier):
    """
        Model that solely finetunes the ViT model from the sequence.
    """
    def __init__(self, input_size=1, hidden_size=512, latent_dim=512, sequence_length=40, num_layers=5, genes=6, delta=64, num_heads=3, init_lr=1e-4, pretrained=True):
        super().__init__(num_classes=2, init_lr=init_lr)
        self.save_hyperparameters()

        self.latent_dim = latent_dim
        self.delta = delta
        self.pretrained = pretrained
        
        if self.pretrained:
            self.vit = models.vit_b_32(weights='IMAGENET1K_V1')
        else:
            self.vit = models.vit_b_32(pretrained=False)

        num_ftrs = self.vit.num_classes
        self.vit_classifier = nn.Linear(num_ftrs, self.latent_dim)  

        neural_net_input = self.latent_dim + self.delta

        # Fusion of image and sequence representations
        self.fc = nn.Sequential(
            nn.Linear(neural_net_input, 512),  # Concatenated vectors are of size 1024 (512 from image + 512 from sequence)
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            # nn.Linear(64, 1),
            # nn.Sigmoid()
            )

        # Prediction heads
        self.heads = nn.ModuleList([nn.Linear(64, 1) for _ in range(num_heads)])

    def forward(self, image, sequence):
        # Image processing
        img_latent = self.vit_classifier(self.vit(image))

        # Calculating delta
        delta_latent = torch.max(sequence, dim=1)[0] - torch.min(sequence, dim=1)[0]
        delta_latent = delta_latent.expand((-1, self.delta))

        # Fusion
        fusion = torch.cat((img_latent, delta_latent), dim=1)
        output = self.fc(fusion)

        # Get predictions for each head
        outputs = torch.stack([torch.sigmoid(head(output)) for head in self.heads], dim=-1)

        return outputs.squeeze()

    def on_validation_epoch_end(self):
        y_hat = torch.cat([o["y_hat"] for o in self.validation_outputs])
        y = torch.cat([o["y"] for o in self.validation_outputs])

        self.log("val_auc", self.auc(y_hat[:,0], y[:,0]), sync_dist=True, prog_bar=True)
        self.log("val_acc", self.accuracy(y_hat[:,0], y[:,0]), sync_dist=True, prog_bar=True)

class GeneFusionHeadsModel(Classifier):
    """
        Model that takes in sequence, image, and gene data and outputs multiple prediction heads (for pred, igi_fp, igi_fn).
    """
    def __init__(self, input_size=1, hidden_size=512, latent_dim=512, sequence_length=40, num_layers=5, genes=6, delta=64, num_heads=3, init_lr=1e-4):
        super().__init__(num_classes=2, init_lr=init_lr)
        self.save_hyperparameters()

        self.latent_dim = latent_dim
        self.delta = delta
        
        self.vit = models.vit_b_32(weights='IMAGENET1K_V1')
        num_ftrs = self.vit.num_classes
        self.vit_classifier = nn.Linear(num_ftrs, self.latent_dim)  # Adjusting to output a 512-dimensional 

        # Sequence processing via LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.hidden_state = (torch.zeros(num_layers, sequence_length, hidden_size), torch.zeros(num_layers, sequence_length, hidden_size))
        # Final fully connected layer to ensure the LSTM output has a size of 512
        self.lstm_fc = nn.Linear(hidden_size, self.latent_dim)

        # Caluclate neural_net input size after appending genes and delta latent
        neural_net_input = self.latent_dim*2 + genes + delta

        # Fusion of image and sequence representations
        self.fc = nn.Sequential(
            nn.Linear(neural_net_input, 512),  # Concatenated vectors are of size 1024 (512 from image + 512 from sequence)
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            # nn.Linear(64, 1),
            # nn.Sigmoid()
            )

        # Prediction heads
        self.heads = nn.ModuleList([nn.Linear(64, 1) for _ in range(num_heads)])

    def forward(self, image, sequence, genes):
        # Image processing
        img_latent = self.vit_classifier(self.vit(image))

        # Sequence processing
        lstm_out, _ = self.lstm(sequence)
        seq_latent = self.lstm_fc(lstm_out[:, -1, :])  # Taking the last output from LSTM for the whole sequence

        # Calculating delta
        delta_latent = torch.max(sequence, dim=1)[0] - torch.min(sequence, dim=1)[0]
        delta_latent = delta_latent.expand((-1, self.delta))

        # Fusion
        fusion = torch.cat((img_latent, seq_latent, genes.squeeze(1), delta_latent), dim=1)
        # fusion = torch.cat((img_latent, seq_latent, genes.squeeze(1), seq_latent_delta), dim=1)
        output = self.fc(fusion)

        # Get predictions for each head
        outputs = torch.stack([torch.sigmoid(head(output)) for head in self.heads], dim=-1)

        return outputs.squeeze()
    
    def on_validation_epoch_end(self):
        y_hat = torch.cat([o["y_hat"] for o in self.validation_outputs])
        y = torch.cat([o["y"] for o in self.validation_outputs])

        self.log("val_auc", self.auc(y_hat[:,0], y[:,0]), sync_dist=True, prog_bar=True)
        self.log("val_acc", self.accuracy(y_hat[:,0], y[:,0]), sync_dist=True, prog_bar=True)
        self.validation_outputs = []

class GeneEnsembleModel(Classifier):
    """
        Model that takes in sequence, image, gene data, and igi call and outputs single prediction head. Note that the FusionModel loaded in must have 3 output heads.
    """
    def __init__(self, input_size, hidden_size, latent_dim, sequence_length, num_layers=5, init_lr=1e-4, fusion_path=None):
        super().__init__(num_classes=2, init_lr=init_lr)
        self.save_hyperparameters()

        self.fusion = FusionModel(input_size, hidden_size, latent_dim, sequence_length, num_layers=num_layers)
        if fusion_path != None:
            self.fusion.load_state_dict(torch.load(fusion_path))

        self.fc = nn.Linear(4, 1)

    def forward(self, image, sequence, genes, igi_call):
        x = self.fusion(image, sequence, genes)
        x = torch.cat(x + [igi_call.view(-1, 1)], dim=1)
        x = torch.sigmoid(self.fc(x))
        return x.squeeze()
    

class ViTFusionModel(nn.Module):
    def __init__(self, input_size, hidden_size, latent_dim, sequence_length, num_layers=5, genes = 6, num_heads=3, delta=16):
        super(ViTFusionModel, self).__init__()

        self.latent_dim = latent_dim
        self.delta = delta
        
        self.vit = models.vit_b_32(weights='IMAGENET1K_V1')
        num_ftrs = self.vit.num_classes
        self.vit_classifier = nn.Linear(num_ftrs, self.latent_dim)  # Adjusting to output a 512-dimensional 

        # Sequence processing via LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.hidden_state = (torch.zeros(num_layers, sequence_length, hidden_size), torch.zeros(num_layers, sequence_length, hidden_size))
        
        # Final fully connected layer to ensure the LSTM output has a size of 512
        self.lstm_fc = nn.Linear(hidden_size, self.latent_dim)

        # Caluclate neural_net input size after appending genes
        neural_net_input = self.latent_dim*2 + genes + delta

        # Fusion of image and sequence representations
        self.fc = nn.Sequential(
            nn.Linear(neural_net_input, 512),  # Concatenated vectors are of size 1024 (512 from image + 512 from sequence)
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
            # nn.Linear(64, 1),
            # nn.Sigmoid()
        )

        # Prediction heads
        self.heads = nn.ModuleList([nn.Linear(64, 1) for _ in range(num_heads)])


    def forward(self, image, sequence, genes):
        # Image processing
        img_latent = self.vit_classifier(self.vit(image))

        # Sequence processing
        lstm_out, _ = self.lstm(sequence)
        seq_latent = self.lstm_fc(lstm_out[:, -1, :])  # Taking the last output from LSTM for the whole sequence

        # Calculating delta
        delta_latent = torch.max(sequence, dim=1)[0] - torch.min(sequence, dim=1)[0]
        delta_latent = delta_latent.expand((-1, self.delta))

        # Fusion
        fusion = torch.cat((img_latent, seq_latent, genes.squeeze(1), delta_latent), dim=1)
        output = self.fc(fusion)

        # Get predictions for each head
        outputs = [torch.sigmoid(head(output)) for head in self.heads]

        return outputs
    

In [6]:
datamodule = ImageSequenceGeneDataModule(curve_dict_path='/media/ssd1/huong/PCR-huong/data/groundtruth_df_curve_dict.pkl',
                             target_df_path='/media/ssd1/huong/PCR-huong/data/groundtruth_df_target_data_split_v2.csv',
                             img_directory='/media/ssd1/huong/PCR-huong/data/curve_imgs_no_axis/',
                             num_workers=32)
testloader = datamodule.test_dataloader()

datamodule = ImageSequenceGeneDataModule(curve_dict_path='/media/ssd1/huong/PCR-huong/data/retest_curve_dict.pkl',
                             target_df_path='/media/ssd1/huong/PCR-huong/data/retest_df_target_data.csv',
                             img_directory='/media/ssd1/huong/PCR-huong/data/curve_imgs_no_axis/',
                             num_workers=32)
retestloader = datamodule.test_dataloader()

datamodule = ImageSequenceGeneDataModule(curve_dict_path='/media/ssd1/huong/PCR-huong/data/chip60_curve_dict.pkl',
                             target_df_path='/media/ssd1/huong/PCR-huong/data/chip60_target_data.csv',
                             img_directory='/media/ssd1/huong/PCR-huong/data/curve_imgs_no_axis/',
                             num_workers=32)
chip60loader = datamodule.test_dataloader()

datamodule = ImageSequenceGeneDataModule(curve_dict_path='/media/ssd1/huong/PCR-huong/data/karlen_curve_dict.pkl',
                             target_df_path='/media/ssd1/huong/PCR-huong/data/karlen_target_data.csv',
                             img_directory='/media/ssd1/huong/PCR-huong/data/curve_imgs_no_axis/',
                             num_workers=32)
karlenloader = datamodule.test_dataloader()

datamodule = ImageSequenceGeneDataModule(curve_dict_path='/media/ssd1/huong/PCR-huong/data/known_curve_dict.pkl',
                             target_df_path='/media/ssd1/huong/PCR-huong/data/known_target_data.csv',
                             img_directory='/media/ssd1/huong/PCR-huong/data/curve_imgs_no_axis/',
                             num_workers=32)
knownloader = datamodule.test_dataloader()


WE ARE USING THE IMAGE SEQUENCE GENE DATASET


100%|██████████| 13949/13949 [00:00<00:00, 47799.07it/s]


WE ARE USING THE IMAGE SEQUENCE GENE DATASET


0it [00:00, ?it/s]
/tmp/ipykernel_2401961/1776985744.py:277: RuntimeWarning: Mean of empty slice.
  self.norm_mean = np.array(mean_list).mean().item()
/home/alpaca/anaconda3/envs/huong-pl/lib/python3.11/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/tmp/ipykernel_2401961/1776985744.py:280: RuntimeWarning: Mean of empty slice.
  self.norm_std = np.array(std_list).mean().item()


WE ARE USING THE IMAGE SEQUENCE GENE DATASET


0it [00:00, ?it/s]
/tmp/ipykernel_2401961/1776985744.py:277: RuntimeWarning: Mean of empty slice.
  self.norm_mean = np.array(mean_list).mean().item()
/home/alpaca/anaconda3/envs/huong-pl/lib/python3.11/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/tmp/ipykernel_2401961/1776985744.py:280: RuntimeWarning: Mean of empty slice.
  self.norm_std = np.array(std_list).mean().item()


WE ARE USING THE IMAGE SEQUENCE GENE DATASET


0it [00:00, ?it/s]
/tmp/ipykernel_2401961/1776985744.py:277: RuntimeWarning: Mean of empty slice.
  self.norm_mean = np.array(mean_list).mean().item()
/home/alpaca/anaconda3/envs/huong-pl/lib/python3.11/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/tmp/ipykernel_2401961/1776985744.py:280: RuntimeWarning: Mean of empty slice.
  self.norm_std = np.array(std_list).mean().item()


WE ARE USING THE IMAGE SEQUENCE GENE DATASET


0it [00:00, ?it/s]
/tmp/ipykernel_2401961/1776985744.py:277: RuntimeWarning: Mean of empty slice.
  self.norm_mean = np.array(mean_list).mean().item()
/home/alpaca/anaconda3/envs/huong-pl/lib/python3.11/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/tmp/ipykernel_2401961/1776985744.py:280: RuntimeWarning: Mean of empty slice.
  self.norm_std = np.array(std_list).mean().item()


#### Sequence 

In [7]:
seq_model = SeqModel().load_from_checkpoint('/media/ssd1/huong/PCR-huong/data/model_checkpoints/experimental/seq_best.ckpt')
seq_model.to(device)
seq_model.eval()

### test
probs, groundtruths, curve_ids = [], [], []
for batch in tqdm(testloader):
    images, seq, gene, labels, ids = batch

    seq = seq.to(device)
    out = seq_model(images, seq, gene)

    probs.append(out[:,0].detach().cpu().numpy())
    groundtruths.append(labels)
    curve_ids.append(ids)
    
seq_test_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
                                 'groundtruth_label': np.concatenate(groundtruths),
                             'outputs': np.concatenate(probs).squeeze()})
seq_test_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/experimental/seq_test_pred_df.csv', index=False)


### retest
probs, groundtruths, curve_ids = [], [], []
for batch in tqdm(retestloader):
    images, seq, gene, labels, ids = batch

    seq = seq.to(device)
    out = seq_model(images, seq, gene)

    probs.append(out[:,0].detach().cpu().numpy())
    groundtruths.append(labels)
    curve_ids.append(ids)
    
seq_retest_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
                                 'groundtruth_label': np.concatenate(groundtruths),
                             'outputs': np.concatenate(probs).squeeze()})
seq_retest_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/experimental/seq_retest_pred_df.csv', index=False)


### chip60
probs, groundtruths, curve_ids = [], [], []
for batch in tqdm(chip60loader):
    images, seq, gene, labels, ids = batch

    seq = seq.to(device)
    out = seq_model(images, seq, gene)

    probs.append(out[:,0].detach().cpu().numpy())
    groundtruths.append(labels)
    curve_ids.append(ids)
    
seq_chip60_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
                                 'groundtruth_label': np.concatenate(groundtruths),
                             'outputs': np.concatenate(probs).squeeze()})
seq_chip60_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/experimental/seq_chip60_pred_df.csv', index=False)



### karlen
probs, groundtruths, curve_ids = [], [], []
for batch in tqdm(karlenloader):
    images, seq, gene, labels, ids = batch

    seq = seq.to(device)
    out = seq_model(images, seq, gene)

    probs.append(out[:,0].detach().cpu().numpy())
    groundtruths.append(labels)
    curve_ids.append(ids)
    
seq_karlen_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
                                 'groundtruth_label': np.concatenate(groundtruths),
                             'outputs': np.concatenate(probs).squeeze()})
seq_karlen_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/experimental/seq_karlen_pred_df.csv', index=False)



### known
probs, groundtruths, curve_ids = [], [], []
for batch in tqdm(knownloader):
    images, seq, gene, labels, ids = batch

    seq = seq.to(device)
    out = seq_model(images, seq, gene)

    probs.append(out[:,0].detach().cpu().numpy())
    groundtruths.append(labels)
    curve_ids.append(ids)
    
seq_known_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
                                 'groundtruth_label': np.concatenate(groundtruths),
                             'outputs': np.concatenate(probs).squeeze()})
seq_known_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/experimental/seq_known_pred_df.csv', index=False)


/home/alpaca/anaconda3/envs/huong-pl/lib/python3.11/site-packages/lightning/pytorch/utilities/migration/utils.py:55: PossibleUserWarning: The loaded checkpoint was produced with Lightning v2.1.1, which is newer than your current Lightning version: v2.0.9.post0
  rank_zero_warn(
100%|██████████| 9/9 [00:01<00:00,  7.96it/s]


### Sequence and gene


In [8]:
seq_gene_model = SeqGeneModel().load_from_checkpoint('/media/ssd1/huong/PCR-huong/data/model_checkpoints/experimental/seq_gene_best.ckpt')
seq_gene_model.to(device)
seq_gene_model.eval()

# ### test
probs, groundtruths, curve_ids = [], [], []
for batch in tqdm(testloader):
    images, seq, gene, labels, ids = batch

    seq = seq.to(device)
    gene = gene.to(device)
    images = images.to(device)
    out = seq_gene_model(images, seq, gene)

    probs.append(out[:,0].detach().cpu().numpy())
    groundtruths.append(labels)
    curve_ids.append(ids)
    
seq_gene_test_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
                                 'groundtruth_label': np.concatenate(groundtruths),
                             'outputs': np.concatenate(probs).squeeze()})
seq_gene_test_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/experimental/seq_gene_test_pred_df.csv', index=False)


### retest
probs, groundtruths, curve_ids = [], [], []
for batch in tqdm(retestloader):
    images, seq, gene, labels, ids = batch

    seq = seq.to(device)
    gene = gene.to(device)
    images = images.to(device)
    out = seq_gene_model(images, seq, gene)

    probs.append(out[:,0].detach().cpu().numpy())
    groundtruths.append(labels)
    curve_ids.append(ids)
    
seq_gene_retest_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
                                 'groundtruth_label': np.concatenate(groundtruths),
                             'outputs': np.concatenate(probs).squeeze()})
seq_gene_retest_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/experimental/seq_gene_retest_pred_df.csv', index=False)

### chip60
probs, groundtruths, curve_ids = [], [], []
for batch in tqdm(chip60loader):
    images, seq, gene, labels, ids = batch

    seq = seq.to(device)
    gene = gene.to(device)
    images = images.to(device)
    out = seq_gene_model(images, seq, gene)

    probs.append(out[:,0].detach().cpu().numpy())
    groundtruths.append(labels)
    curve_ids.append(ids)
    
seq_gene_chip60_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
                                 'groundtruth_label': np.concatenate(groundtruths),
                             'outputs': np.concatenate(probs).squeeze()})
seq_gene_chip60_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/experimental/seq_gene_chip60_pred_df.csv', index=False)

### karlen
probs, groundtruths, curve_ids = [], [], []
for batch in tqdm(karlenloader):
    images, seq, gene, labels, ids = batch

    seq = seq.to(device)
    gene = gene.to(device)
    images = images.to(device)
    out = seq_gene_model(images, seq, gene)

    probs.append(out[:,0].detach().cpu().numpy())
    groundtruths.append(labels)
    curve_ids.append(ids)
    
seq_gene_karlen_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
                                 'groundtruth_label': np.concatenate(groundtruths),
                             'outputs': np.concatenate(probs).squeeze()})
seq_gene_karlen_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/experimental/seq_gene_karlen_pred_df.csv', index=False)

### known
probs, groundtruths, curve_ids = [], [], []
for batch in tqdm(knownloader):
    images, seq, gene, labels, ids = batch

    seq = seq.to(device)
    gene = gene.to(device)
    images = images.to(device)
    out = seq_gene_model(images, seq, gene)

    probs.append(out[:,0].detach().cpu().numpy())
    groundtruths.append(labels)
    curve_ids.append(ids)
    
seq_gene_known_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
                                 'groundtruth_label': np.concatenate(groundtruths),
                             'outputs': np.concatenate(probs).squeeze()})
seq_gene_known_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/experimental/seq_gene_known_pred_df.csv', index=False)



100%|██████████| 9/9 [00:01<00:00,  7.98it/s]


### Sequence Curve and gene

In [9]:
seq_curve_gene_model = SeqCurveGeneModel().load_from_checkpoint('/media/ssd1/huong/PCR-huong/data/model_checkpoints/experimental/seq_curve_gene_best.ckpt')
seq_curve_gene_model.to(device)
seq_curve_gene_model.eval()

### test
probs, groundtruths, curve_ids = [], [], []
for batch in tqdm(testloader):
    images, seq, gene, labels, ids = batch

    seq = seq.to(device)
    gene = gene.to(device)
    images = images.to(device)
    out = seq_curve_gene_model(images, seq, gene)

    probs.append(out[:,0].detach().cpu().numpy())
    groundtruths.append(labels)
    curve_ids.append(ids)
    
seq_curve_gene_test_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
                                 'groundtruth_label': np.concatenate(groundtruths),
                             'outputs': np.concatenate(probs).squeeze()})
seq_curve_gene_test_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/experimental/seq_curve_gene_test_pred_df.csv', index=False)


### retest
probs, groundtruths, curve_ids = [], [], []
for batch in tqdm(retestloader):
    images, seq, gene, labels, ids = batch

    seq = seq.to(device)
    gene = gene.to(device)
    images = images.to(device)
    out = seq_curve_gene_model(images, seq, gene)

    probs.append(out[:,0].detach().cpu().numpy())
    groundtruths.append(labels)
    curve_ids.append(ids)
    
seq_curve_gene_retest_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
                                 'groundtruth_label': np.concatenate(groundtruths),
                             'outputs': np.concatenate(probs).squeeze()})
seq_curve_gene_retest_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/experimental/seq_curve_gene_retest_pred_df.csv', index=False)


### chip60
probs, groundtruths, curve_ids = [], [], []
for batch in tqdm(chip60loader):
    images, seq, gene, labels, ids = batch

    seq = seq.to(device)
    gene = gene.to(device)
    images = images.to(device)
    out = seq_curve_gene_model(images, seq, gene)

    probs.append(out[:,0].detach().cpu().numpy())
    groundtruths.append(labels)
    curve_ids.append(ids)
    
seq_curve_gene_chip60_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
                                 'groundtruth_label': np.concatenate(groundtruths),
                             'outputs': np.concatenate(probs).squeeze()})
seq_curve_gene_chip60_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/experimental/seq_curve_gene_chip60_pred_df.csv', index=False)


### karlen
probs, groundtruths, curve_ids = [], [], []
for batch in tqdm(karlenloader):
    images, seq, gene, labels, ids = batch

    seq = seq.to(device)
    gene = gene.to(device)
    images = images.to(device)
    out = seq_curve_gene_model(images, seq, gene)

    probs.append(out[:,0].detach().cpu().numpy())
    groundtruths.append(labels)
    curve_ids.append(ids)
    
seq_curve_gene_karlen_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
                                 'groundtruth_label': np.concatenate(groundtruths),
                             'outputs': np.concatenate(probs).squeeze()})
seq_curve_gene_karlen_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/experimental/seq_curve_gene_karlen_pred_df.csv', index=False)


### known
probs, groundtruths, curve_ids = [], [], []
for batch in tqdm(knownloader):
    images, seq, gene, labels, ids = batch

    seq = seq.to(device)
    gene = gene.to(device)
    images = images.to(device)
    out = seq_curve_gene_model(images, seq, gene)

    probs.append(out[:,0].detach().cpu().numpy())
    groundtruths.append(labels)
    curve_ids.append(ids)
    
seq_curve_gene_known_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
                                 'groundtruth_label': np.concatenate(groundtruths),
                             'outputs': np.concatenate(probs).squeeze()})
seq_curve_gene_known_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/experimental/seq_curve_gene_known_pred_df.csv', index=False)


100%|██████████| 9/9 [00:01<00:00,  6.00it/s]


#### Image without range model

In [10]:
img_model = CurveShapeModel().load_from_checkpoint('/media/ssd1/huong/PCR-huong/data/model_checkpoints/experimental/epoch=37-step=2090.ckpt')

probs, groundtruths, curve_ids = [], [], []

img_model.to(device)
img_model.eval()
for batch in tqdm(testloader):
    images, seq, gene, labels, ids = batch

    images = images.to(device)
    out = img_model(images)

    probs.append(out[:,0].detach().cpu().numpy())
    groundtruths.append(labels)
    curve_ids.append(ids)

img_test_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
                                 'groundtruth_label': np.concatenate(groundtruths),
                             'outputs': np.concatenate(probs).squeeze()})
img_test_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/experimental/Img_wo_range_test_pred_df.csv', index=False)


### retest
probs, groundtruths, curve_ids = [], [], []
img_model.to(device)
img_model.eval()
for batch in tqdm(retestloader):
    images, seq, gene, labels, ids = batch

    images = images.to(device)
    out = img_model(images)

    probs.append(out[:,0].detach().cpu().numpy())
    groundtruths.append(labels)
    curve_ids.append(ids)

img_retest_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
                                   'groundtruth_label': np.concatenate(groundtruths),
                             'outputs': np.concatenate(probs).squeeze()})
img_retest_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/experimental/Img_wo_range_retest_pred_df.csv', index=False)


### chip60
probs, groundtruths, curve_ids = [], [], []
img_model.to(device)
img_model.eval()
for batch in tqdm(chip60loader):
    images, seq, gene, labels, ids = batch

    images = images.to(device)
    out = img_model(images)

    probs.append(out[:,0].detach().cpu().numpy())
    groundtruths.append(labels)
    curve_ids.append(ids)

img_chip60_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
                                   'groundtruth_label': np.concatenate(groundtruths),
                             'outputs': np.concatenate(probs).squeeze()})
img_chip60_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/experimental/Img_wo_range_chip60_pred_df.csv', index=False)


### karlen
probs, groundtruths, curve_ids = [], [], []
img_model.to(device)
img_model.eval()
for batch in tqdm(karlenloader):
    images, seq, gene, labels, ids = batch

    images = images.to(device)
    out = img_model(images)

    probs.append(out[:,0].detach().cpu().numpy())
    groundtruths.append(labels)
    curve_ids.append(ids)

img_karlen_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
                                   'groundtruth_label': np.concatenate(groundtruths),
                             'outputs': np.concatenate(probs).squeeze()})
img_karlen_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/experimental/Img_wo_range_karlen_pred_df.csv', index=False)


### known
probs, groundtruths, curve_ids = [], [], []
img_model.to(device)
img_model.eval()
for batch in tqdm(knownloader):
    images, seq, gene, labels, ids = batch

    images = images.to(device)
    out = img_model(images)

    probs.append(out[:,0].detach().cpu().numpy())
    groundtruths.append(labels)
    curve_ids.append(ids)

img_known_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids),
                                   'groundtruth_label': np.concatenate(groundtruths),
                             'outputs': np.concatenate(probs).squeeze()})
img_known_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/experimental/Img_wo_range_known_pred_df.csv', index=False)




100%|██████████| 9/9 [00:01<00:00,  5.83it/s]


### VIT Fusion

In [11]:
with open('/media/ssd1/huong/PCR-huong/data/retest_curve_dict.pkl', 'rb') as file:
    retest_curve_dict = pkl.load(file)
retest_df = pd.read_csv('/media/ssd1/huong/PCR-huong/data/retest_df_target_data.csv')

retestVit_dataset = ViTFusionDataset(retest_curve_dict, retest_df,
                                     img_directory = '/media/ssd1/huong/PCR-huong/data/curve_imgs/', 
                                     split='test', sequence_len=40,
                                    mean=94477.0057018847, std = 155626.8370536778)
retestVit_loader = DataLoader(retestVit_dataset, batch_size=32, pin_memory=True, shuffle=True)


with open('/media/ssd1/huong/PCR-huong/data/groundtruth_df_curve_dict.pkl', 'rb') as file:
    test_curve_dict = pkl.load(file)
test_df = pd.read_csv('/media/ssd1/huong/PCR-huong/data/groundtruth_df_target_data_split_v2.csv')

testVit_dataset = ViTFusionDataset(test_curve_dict, test_df,
                                   img_directory = '/media/ssd1/huong/PCR-huong/data/curve_imgs/', 
                                   split='test', sequence_len=40,
                                    mean=94477.0057018847, std = 155626.8370536778)
testVit_loader = DataLoader(testVit_dataset, batch_size=32, pin_memory=True, shuffle=True)

with open('/media/ssd1/huong/PCR-huong/data/known_curve_dict.pkl', 'rb') as file:
    known_curve_dict = pkl.load(file)
known_df = pd.read_csv('/media/ssd1/huong/PCR-huong/data/known_target_data.csv')

knownVit_dataset = ViTFusionDataset(known_curve_dict, known_df,
                                    img_directory = '/media/ssd1/huong/PCR-huong/data/curve_imgs/', 
                                   split='test', sequence_len=40,
                                    mean=94477.0057018847, std = 155626.8370536778)
knownVit_loader = DataLoader(knownVit_dataset, batch_size=32, pin_memory=True, shuffle=True)

with open('/media/ssd1/huong/PCR-huong/data/chip60_curve_dict.pkl', 'rb') as file:
    chip_curve_dict = pkl.load(file)
chip_df = pd.read_csv('/media/ssd1/huong/PCR-huong/data/chip60_target_data.csv')

chipVit_dataset = ViTFusionDataset(chip_curve_dict, chip_df,
                                    img_directory = '/media/ssd1/huong/PCR-huong/data/curve_imgs/', 
                                   split='test', sequence_len=40,
                                    mean=94477.0057018847, std = 155626.8370536778)
chipVit_loader = DataLoader(chipVit_dataset, batch_size=32, pin_memory=True, shuffle=True)

with open('/media/ssd1/huong/PCR-huong/data/karlen_curve_dict.pkl', 'rb') as file:
    karlen_curve_dict = pkl.load(file)
karlen_df = pd.read_csv('/media/ssd1/huong/PCR-huong/data/karlen_target_data.csv')

karlenVit_dataset = ViTFusionDataset(karlen_curve_dict, karlen_df,
                                    img_directory = '/media/ssd1/huong/PCR-huong/data/curve_imgs/', 
                                   split='test', sequence_len=40,
                                    mean=94477.0057018847, std = 155626.8370536778)
karlenVit_loader = DataLoader(karlenVit_dataset, batch_size=32, pin_memory=True, shuffle=True)




In [12]:
fusion_model = ViTFusionModel(input_size=1, hidden_size=512, latent_dim=512, sequence_length=40, num_layers=3, genes=6,delta=64)
fusion_model.load_state_dict(torch.load('/media/ssd1/huong/PCR-huong/data/model_checkpoints/fusion_model/10_27_fusion_model_vit_delta64.pth'))
fusion_model.to(device)
fusion_model.eval()

probs, groundtruths, curve_ids = [], [], []

for batch in tqdm(retestVit_loader):
    images, seq, gene, labels, ids = batch

    images = images.to(device)
    seq = seq.to(device).unsqueeze(2)
    gene = gene.to(device)

    out = fusion_model(images, seq, gene)
    probs.append(out[0].detach().cpu().numpy())
    groundtruths.append(labels[0])
    curve_ids.append(ids)

fusion_retest_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids), 
                             'outputs': np.concatenate(probs).squeeze(),
                             'groundtruth_label': np.concatenate(groundtruths)})
fusion_retest_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/vitfusion_retest_pred_df.csv', index = False)


probs, groundtruths, curve_ids = [], [], []
for batch in tqdm(testVit_loader):
    images, seq, gene, labels, ids = batch

    images = images.to(device)
    seq = seq.to(device).unsqueeze(2)
    gene = gene.to(device)

    out = fusion_model(images, seq, gene)
    probs.append(out[0].detach().cpu().numpy())
    groundtruths.append(labels[0])
    curve_ids.append(ids)

fusion_test_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids), 
                             'outputs': np.concatenate(probs).squeeze(),
                             'groundtruth_label': np.concatenate(groundtruths)})
fusion_test_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/vitfusion_test_pred_df.csv', index = False)



probs, groundtruths, curve_ids = [], [], []
for batch in tqdm(knownVit_loader):
    images, seq, gene, labels, ids = batch

    images = images.to(device)
    seq = seq.to(device).unsqueeze(2)
    gene = gene.to(device)

    out = fusion_model(images, seq, gene)
    probs.append(out[0].detach().cpu().numpy())
    groundtruths.append(labels[0])
    curve_ids.append(ids)

fusion_known_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids), 
                             'outputs': np.concatenate(probs).squeeze(),
                             'groundtruth_label': np.concatenate(groundtruths)})
fusion_known_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/vitfusion_known_pred_df.csv', index = False)



probs, groundtruths, curve_ids = [], [], []
for batch in tqdm(chipVit_loader):
    images, seq, gene, labels, ids = batch

    images = images.to(device)
    seq = seq.to(device).unsqueeze(2)
    gene = gene.to(device)

    out = fusion_model(images, seq, gene)
    probs.append(out[0].detach().cpu().numpy())
    groundtruths.append(labels[0])
    curve_ids.append(ids)

fusion_chip60_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids), 
                             'outputs': np.concatenate(probs).squeeze(),
                             'groundtruth_label': np.concatenate(groundtruths)})
fusion_chip60_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/vitfusion_chip_pred_df.csv', index = False)



probs, groundtruths, curve_ids = [], [], []
for batch in tqdm(karlenVit_loader):
    images, seq, gene, labels, ids = batch

    images = images.to(device)
    seq = seq.to(device).unsqueeze(2)
    gene = gene.to(device)

    out = fusion_model(images, seq, gene)
    probs.append(out[0].detach().cpu().numpy())
    groundtruths.append(labels[0])
    curve_ids.append(ids)

fusion_karlen_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids), 
                             'outputs': np.concatenate(probs).squeeze(),
                             'groundtruth_label': np.concatenate(groundtruths)})
fusion_karlen_pred_df.to_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/vitfusion_karlen_pred_df.csv', index = False)

100%|██████████| 9/9 [00:01<00:00,  5.11it/s]


### Evaluation

In [13]:
img_test_auc = roc_auc_score(img_test_pred_df.groundtruth_label, img_test_pred_df.outputs)
print('AUC of image only model on test set:', img_test_auc)
img_retest_auc = roc_auc_score(img_retest_pred_df.groundtruth_label, img_retest_pred_df.outputs)
print('AUC of image only model on retest set:', img_retest_auc)
img_chip60_auc = roc_auc_score(img_chip60_pred_df.groundtruth_label, img_chip60_pred_df.outputs)
print('AUC of image only model on chip60 set:', img_chip60_auc)
img_known_auc = roc_auc_score(img_known_pred_df.groundtruth_label, img_known_pred_df.outputs)
print('AUC of image only model on known set:', img_known_auc)
# img_karlen_auc = roc_auc_score(img_karlen_pred_df.groundtruth_label, img_karlen_pred_df.outputs)
# print('AUC of image only model on karlen set:', img_karlen_auc)
# img_known_auc = roc_auc_score(img_known_pred_df.groundtruth_label, img_known_pred_df.outputs)
# print('AUC of image only model on known set:', img_known_auc)
print('\n')
seq_test_auc = roc_auc_score(seq_test_pred_df.groundtruth_label, seq_test_pred_df.outputs)
print('AUC of sequence only model on test set:', seq_test_auc)
seq_retest_auc = roc_auc_score(seq_retest_pred_df.groundtruth_label, seq_retest_pred_df.outputs)
print('AUC of sequence only model on retest set:', seq_retest_auc)
seq_chip60_auc = roc_auc_score(seq_chip60_pred_df.groundtruth_label, seq_chip60_pred_df.outputs)
print('AUC of sequence only model on chip60 set:', seq_chip60_auc)
seq_known_auc = roc_auc_score(seq_known_pred_df.groundtruth_label, seq_known_pred_df.outputs)
print('AUC of sequence only model on known set:', seq_known_auc)
# seq_karlen_auc = roc_auc_score(seq_karlen_pred_df.groundtruth_label, seq_karlen_pred_df.outputs)
# print('AUC of sequence only model on karlen set:', seq_karlen_auc)
# seq_known_auc = roc_auc_score(seq_known_pred_df.groundtruth_label, seq_known_pred_df.outputs)
# print('AUC of sequence only model on known set:', seq_known_auc)
print('\n')
seq_gene_test_auc = roc_auc_score(seq_gene_test_pred_df.groundtruth_label, seq_gene_test_pred_df.outputs)
print('AUC of sequence gene model on test set:', seq_gene_test_auc)
seq_gene_retest_auc = roc_auc_score(seq_gene_retest_pred_df.groundtruth_label, seq_gene_retest_pred_df.outputs)
print('AUC of sequence gene model on retest set:', seq_gene_retest_auc)
seq_gene_chip60_auc = roc_auc_score(seq_gene_chip60_pred_df.groundtruth_label, seq_gene_chip60_pred_df.outputs)
print('AUC of sequence gene model on chip60 set:', seq_gene_chip60_auc)
seq_gene_known_auc = roc_auc_score(seq_gene_known_pred_df.groundtruth_label, seq_gene_known_pred_df.outputs)
print('AUC of sequence gene model on known set:', seq_gene_known_auc)
# seq_gene_karlen_auc = roc_auc_score(seq_gene_karlen_pred_df.groundtruth_label, seq_gene_karlen_pred_df.outputs)
# print('AUC of sequence gene model on karlen set:', seq_gene_karlen_auc)
# seq_gene_known_auc = roc_auc_score(seq_gene_known_pred_df.groundtruth_label, seq_gene_known_pred_df.outputs)
# print('AUC of sequence gene model on known set:', seq_gene_known_auc)
print('\n')
seq_curve_gene_test_auc = roc_auc_score(seq_curve_gene_test_pred_df.groundtruth_label, seq_curve_gene_test_pred_df.outputs)
print('AUC of sequence gene curve model on test set:', seq_curve_gene_test_auc)
seq_curve_gene_retest_auc = roc_auc_score(seq_curve_gene_retest_pred_df.groundtruth_label, seq_curve_gene_retest_pred_df.outputs)
print('AUC of sequence gene curve model on retest set:', seq_curve_gene_retest_auc)
seq_curve_gene_chip60_auc = roc_auc_score(seq_curve_gene_chip60_pred_df.groundtruth_label, seq_curve_gene_chip60_pred_df.outputs)
print('AUC of sequence gene curve model on chip60 set:', seq_curve_gene_chip60_auc)
seq_curve_gene_known_auc = roc_auc_score(seq_curve_gene_known_pred_df.groundtruth_label, seq_curve_gene_known_pred_df.outputs)
print('AUC of sequence gene curve model on known set:', seq_curve_gene_known_auc)
# seq_curve_gene_karlen_auc = roc_auc_score(seq_curve_gene_karlen_pred_df.groundtruth_label, seq_curve_gene_karlen_pred_df.outputs)
# print('AUC of sequence gene curve model on karlen set:', seq_curve_gene_karlen_auc)
# seq_curve_gene_known_auc = roc_auc_score(seq_curve_gene_known_pred_df.groundtruth_label, seq_curve_gene_known_pred_df.outputs)
# print('AUC of sequence gene curve model on known set:', seq_curve_gene_known_auc)

print('\n')
fusion_test_auc = roc_auc_score(fusion_test_pred_df.groundtruth_label, fusion_test_pred_df.outputs)
print('AUC of ViT fusion model on test set:', fusion_test_auc)
fusion_retest_auc = roc_auc_score(fusion_retest_pred_df.groundtruth_label, fusion_retest_pred_df.outputs)
print('AUC of ViT fusion model on retest set:', fusion_retest_auc)
fusion_chip60_auc = roc_auc_score(fusion_chip60_pred_df.groundtruth_label, fusion_chip60_pred_df.outputs)
print('AUC of ViT fusion model on chip60 set:', fusion_chip60_auc)
fusion_known_auc = roc_auc_score(fusion_known_pred_df.groundtruth_label, fusion_known_pred_df.outputs)
print('AUC of ViT fusion model on known set:', fusion_known_auc)
# seq_curve_gene_karlen_auc = roc_auc_score(seq_curve_gene_karlen_pred_df.groundtruth_label, seq_curve_gene_karlen_pred_df.outputs)
# print('AUC of ViT fusion model on karlen set:', seq_curve_gene_karlen_auc)
# seq_curve_gene_known_auc = roc_auc_score(seq_curve_gene_known_pred_df.groundtruth_label, seq_curve_gene_known_pred_df.outputs)
# print('AUC of ViT fusion model on known set:', seq_curve_gene_known_auc)





AUC of image only model on test set: 0.9835164167988861
AUC of image only model on retest set: 0.8531415660412816
AUC of image only model on chip60 set: 0.46428571428571425
AUC of image only model on known set: 0.952121310763889


AUC of sequence only model on test set: 0.9945482604854264
AUC of sequence only model on retest set: 0.8991766675073003
AUC of sequence only model on chip60 set: 1.0
AUC of sequence only model on known set: 0.9720323350694444


AUC of sequence gene model on test set: 0.9980473538377564
AUC of sequence gene model on retest set: 0.9479394579275734
AUC of sequence gene model on chip60 set: 0.9821428571428572
AUC of sequence gene model on known set: 0.9759114583333333


AUC of sequence gene curve model on test set: 0.9976923272628029
AUC of sequence gene curve model on retest set: 0.9402902062585816
AUC of sequence gene curve model on chip60 set: 0.7678571428571429
AUC of sequence gene curve model on known set: 0.9591200086805555


AUC of ViT fusion model on test

In [21]:
qpcr_chip = pd.read_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/qpcr_chip60_pred.csv')

qpcr_known = pd.read_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/qpcr_known_pred.csv')

qpcr_retest = pd.read_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/qpcr_retest_pred.csv')

qpcr_groundtruth = pd.read_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/qpcr_groundtruth_pred.csv')
qpcr_test = qpcr_groundtruth[qpcr_groundtruth.split == 'test']

qpcr_karlen1 = pd.read_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/qpcr_karlen1_pred.csv')
qpcr_karlen1['file'] = 'karlen1'
qpcr_karlen2 = pd.read_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/qpcr_karlen2_pred.csv')
qpcr_karlen2['file'] = 'karlen2'
qpcr_karlen3 = pd.read_csv('/media/ssd1/huong/PCR-huong/data/model_outputs/qpcr_karlen3_pred.csv')
qpcr_karlen3['file'] = 'karlen3'

qpcr_karlen = pd.concat([qpcr_karlen1, qpcr_karlen2, qpcr_karlen3])



In [22]:
print('Number of NA curves in test:', qpcr_test[qpcr_test.prob.isna()].shape[0])
non_na_qpcr_test = qpcr_test[~qpcr_test.eff.isna()]
print('Number of NA curves in chip:', qpcr_chip[qpcr_chip.eff.isna()].shape[0])
print('Number of NA curves in karlen:', qpcr_karlen[qpcr_karlen.eff.isna()].shape[0])
print('Number of NA curves in known:', qpcr_known[qpcr_known.eff.isna()].shape[0])
print('Number of NA curves in retest:', qpcr_retest[qpcr_retest.eff.isna()].shape[0])
non_na_qpcr_retest = qpcr_retest[~qpcr_retest.eff.isna()]

Number of NA curves in test: 10
Number of NA curves in chip: 0
Number of NA curves in karlen: 0
Number of NA curves in known: 0
Number of NA curves in retest: 18


In [24]:
qpcr_test_auc = roc_auc_score(non_na_qpcr_test.groundtruth_label, non_na_qpcr_test.prob)
print('AUC of qpcR model on test set:', qpcr_test_auc)
qpcr_chip_auc = roc_auc_score(qpcr_chip.groundtruth_label, qpcr_chip.prob)
print('AUC of qpcR model on chip set:', qpcr_chip_auc)
qpcr_retest_auc = roc_auc_score(non_na_qpcr_retest.groundtruth_label, non_na_qpcr_retest.prob)
print('AUC of qpcR model on retest set:', qpcr_retest_auc)
qpcr_known_auc = roc_auc_score(qpcr_known.groundtruth_label, qpcr_known.prob)
print('AUC of qpcR model on known set:', qpcr_known_auc)
# qpcr_karlen_auc = roc_auc_score(qpcr_karlen.groundtruth_label, qpcr_karlen.prob)
# print('AUC of qpcR model on test set:', qpcr_test_auc)

AUC of qpcR model on test set: 0.981763867477239
AUC of qpcR model on chip set: 0.5
AUC of qpcR model on retest set: 0.8239072588585228
AUC of qpcR model on known set: 0.9379340277777779


In [52]:
best_acc_thres = 0.5657657657657658

fusion_karlen_pred_df['pred'] = 1*(fusion_karlen_pred_df.outputs >= best_acc_thres)
fusion_karlen_tn, fusion_karlen_fp, fusion_karlen_fn, fusion_karlen_tp = confusion_matrix(fusion_karlen_pred_df.groundtruth_label, 
                                                                                          fusion_karlen_pred_df.pred).ravel()
fusion_karlen_acc = (fusion_karlen_tp+fusion_karlen_tn)/(fusion_karlen_tp+fusion_karlen_tn+fusion_karlen_fp+fusion_karlen_fn)
print(f'Fusion karlen accuracy: {fusion_karlen_acc}')
print(f'Fusion karlen TP = {fusion_karlen_tp}; TN = {fusion_karlen_tn}; FP = {fusion_karlen_fp}; FN = {fusion_karlen_fn}')

# qpcr_karlen['pred'] = 1*(qpcr_karlen.prob >= best_acc_thres)
qpcr_karlen_tn, qpcr_karlen_fp, qpcr_karlen_fn, qpcr_karlen_tp = 0, 0, 0, 272
qpcr_karlen_acc = (qpcr_karlen_tp+qpcr_karlen_tn)/(qpcr_karlen_tp+qpcr_karlen_tn+qpcr_karlen_fp+qpcr_karlen_fn)
print(f'qpcR karlen accuracy: {qpcr_karlen_acc}')
print(f'qpcR karlen TP = {qpcr_karlen_tp}; TN = {qpcr_karlen_tn}; FP = {qpcr_karlen_fp}; FN = {qpcr_karlen_fn}')

img_karlen_pred_df['pred'] = 1*(img_karlen_pred_df.outputs >= best_acc_thres)
img_karlen_tn, img_karlen_fp, img_karlen_fn, img_karlen_tp = 0, 0, 0, 272
img_karlen_acc = (img_karlen_tp+img_karlen_tn)/(img_karlen_tp+img_karlen_tn+img_karlen_fp+img_karlen_fn)
print(f'Image only karlen accuracy: {img_karlen_acc}')
print(f'Image only karlen TP = {img_karlen_tp}; TN = {img_karlen_tn}; FP = {img_karlen_fp}; FN = {img_karlen_fn}')

seq_karlen_pred_df['pred'] = 1*(seq_karlen_pred_df.outputs >= best_acc_thres)
seq_karlen_tn, seq_karlen_fp, seq_karlen_fn, seq_karlen_tp = 0, 0, 0, 272
seq_karlen_acc = (seq_karlen_tp+seq_karlen_tn)/(seq_karlen_tp+seq_karlen_tn+seq_karlen_fp+seq_karlen_fn)
print(f'Sequence only karlen accuracy: {seq_karlen_acc}')
print(f'Sequence only karlen TP = {seq_karlen_tp}; TN = {seq_karlen_tn}; FP = {seq_karlen_fp}; FN = {seq_karlen_fn}')


seq_curve_gene_karlen_pred_df['pred'] = 1*(seq_curve_gene_karlen_pred_df.outputs >= best_acc_thres)
seq_curve_gene_karlen_tn, seq_curve_gene_karlen_fp, seq_curve_gene_karlen_fn, seq_curve_gene_karlen_tp = 0, 0, 0, 272
seq_curve_gene_karlen_acc = (seq_curve_gene_karlen_tp+seq_curve_gene_karlen_tn)/(seq_curve_gene_karlen_tp+seq_curve_gene_karlen_tn+seq_curve_gene_karlen_fp+seq_curve_gene_karlen_fn)
print(f'Sequence curve gene karlen accuracy: {seq_curve_gene_karlen_acc}')
print(f'Sequence curve gene karlen TP = {seq_curve_gene_karlen_tp}; TN = {seq_curve_gene_karlen_tn}; FP = {seq_curve_gene_karlen_fp}; FN = {seq_curve_gene_karlen_fn}')



seq_gene_karlen_pred_df['pred'] = 1*(seq_gene_karlen_pred_df.outputs >= best_acc_thres)
seq_gene_karlen_tn, seq_gene_karlen_fp, seq_gene_karlen_fn, seq_gene_karlen_tp = 0, 0, 0, 272
seq_gene_karlen_acc = (seq_gene_karlen_tp+seq_gene_karlen_tn)/(seq_gene_karlen_tp+seq_gene_karlen_tn+seq_gene_karlen_fp+seq_gene_karlen_fn)
print(f'Sequence gene karlen accuracy: {seq_gene_karlen_acc}')
print(f'Sequence gene karlen TP = {seq_gene_karlen_tp}; TN = {seq_gene_karlen_tn}; FP = {seq_gene_karlen_fp}; FN = {seq_gene_karlen_fn}')


Fusion karlen accuracy: 0.8529411764705882
Fusion karlen TP = 232; TN = 0; FP = 0; FN = 40
qpcR karlen accuracy: 1.0
qpcR karlen TP = 272; TN = 0; FP = 0; FN = 0
Image only karlen accuracy: 1.0
Image only karlen TP = 272; TN = 0; FP = 0; FN = 0
Sequence only karlen accuracy: 1.0
Sequence only karlen TP = 272; TN = 0; FP = 0; FN = 0
Sequence curve gene karlen accuracy: 1.0
Sequence curve gene karlen TP = 272; TN = 0; FP = 0; FN = 0
Sequence gene karlen accuracy: 1.0
Sequence gene karlen TP = 272; TN = 0; FP = 0; FN = 0


In [50]:
fusion_karlen_pred_df

,curve_idx,outputs,groundtruth_label,pred
0,F1.50.2_karlen2,1.000000,1,1
1,F1.100.2_karlen2,1.000000,1,1
2,F1.1.1_karlen1,0.894196,1,1
3,F3.10.2_karlen1,0.899401,1,1
4,F1.1.5_karlen3,1.000000,1,1
...,...,...,...,...
267,F4.1.4_karlen1,0.849127,1,1
268,F4.50.5_karlen1,0.115801,1,0
269,F2.1.5_karlen3,1.000000,1,1
270,F2.10.2_karlen3,1.000000,1,1


In [43]:
confusion_matrix(seq_gene_karlen_pred_df.groundtruth_label, seq_gene_karlen_pred_df.pred).ravel()

AttributeError: 'DataFrame' object has no attribute 'pred'

In [38]:
img_karlen_pred_df

,curve_idx,groundtruth_label,outputs,pred
0,F1.1.1_karlen1,1.0,0.999758,1
1,F1.1.2_karlen1,1.0,0.999769,1
2,F1.1.3_karlen1,1.0,0.999762,1
3,F1.1.4_karlen1,1.0,0.999775,1
4,F1.1.5_karlen1,1.0,0.999733,1
...,...,...,...,...
267,F4.100.5_karlen3,1.0,0.999798,1
268,F4.1000.1_karlen3,1.0,0.999745,1
269,F4.1000.2_karlen3,1.0,0.999747,1
270,F4.1000.3_karlen3,1.0,0.999723,1
